# 복수의 LLM 요청을 비동기 병렬로 처리
- 비동기(asyncio) : async 처리, asyncio vs multiprocessing : https://cafe.daum.net/flowlife/RUrO/167
---

### web 크롤링(3개) -> LLM 요약 -> ChromaDB에 저장 -> RAG검색 -> LLM 최총 답변

In [23]:
!pip install openai sentence-transformers chromadb python-dotenv
!pip install requests beautifulsoup4

In [24]:
import asyncio # 비동기 처리 라이브러리 로딩

# async 비동기 대상 함수 처리 예시
async def hello():
  print('시작')
  await asyncio.sleep(2)
  print('2초후 실행')

# asyncio.run(hello())# - VScode예시
await hello()

시작
2초후 실행


In [25]:
import asyncio # 비동기 처리 라이브러리 로딩
import os
import textwrap # 긴 문자열을 보기좋게 줄바꿈이 가능한 모듈

from openai import OpenAI
from openai import AsyncOpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import PersistentClient

import requests
from bs4 import BeautifulSoup

from dotenv import load_dotenv

load_dotenv()

URLS=[
    "https://ko.wikipedia.org/wiki/김치찌개",
    "https://ko.wikipedia.org/wiki/인공지능",
    "https://ko.wikipedia.org/wiki/LCK"
]

GPT_MODEL = "gpt-4o-mini"
CHROMA_PATH = './wiki_async'
COLLECTION_NAME = "wiki_docs"
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
  raise RuntimeError("OPENAI_API_KEY가 없음")

# 동기 방식
client = OpenAI(api_key=OPENAI_API_KEY)

# 비동기 방식
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma = PersistentClient(path = CHROMA_PATH)
collection = chroma.get_or_create_collection(COLLECTION_NAME)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [26]:
# 위키백과 URL에서 문서 읽기 함수
# 일반 함수
def fetch_wiki(url:str, max_chars:int=3000) -> str:
  headers = {"User-Agent":"Mozilla/5.0"}
  response = requests.get(url, headers=headers)
  print(f'[FETCH] {url} -> {response.status_code}')

  if response.status_code != 200:
    return ""

  soup = BeautifulSoup(response.text, 'html.parser')
  paragraphs = [
      p.get_text(strip=True) for p in soup.find_all('p') if p.get_text(strip=True)
  ]

  text = '\n'.join(paragraphs)
  return text[:max_chars]

In [27]:
# LLM에게 비동기로 요약을 요청하는 함수 생성
# 비동기 함수 생성
async def summarize_async(url:str, text:str) -> str:
  if not text:
    return ""

  prompt = f"""
  너는 한국어 문서를 잘 요약하는 전문가야

  아래 문서를 핵심 개념, 중요 사실, 특징 중심으로 10문장 이내로 요약해줘

  URL:
  {url}

  문서 내용:
  {text}
  """

  response = await async_client.responses.create(
      model = GPT_MODEL,
      input = prompt,
      temperature= 0
  )
  return response.output_text.strip()

In [28]:
async def build_db_async():
  texts = [] # URL과 본문 텍스트를 저장

  for url in URLS:
    text = fetch_wiki(url) # wiki본문 추출
    texts.append((url, text)) # 수정된 부분: (url, text) 튜플을 추가

  tasks = [
      summarize_async(url, text) for url, text in texts
  ]

  # 여러 LLM요약 요청을 동시에 실행하고 그 결과를 리스트로 받기
  summarizes = await asyncio.gather(*tasks) # tasks가 여러개라 *이 들어감

  # ChromaDB에 저장할 저장공간 list 생성
  docs = []
  ids = []
  metadatas = []

  for i ,((url, _), summary) in enumerate(zip(texts, summarizes)):
    if not summary:
      continue
    docs.append(summary)
    ids.append(f'doc_{i}')
    metadatas.append({"url":url})

  if not docs:
    print('[알림] 저장할 문서가 없음')
    return

  embeddings = embedder.encode(docs).tolist()
  collection.upsert(
      ids = ids,
      documents = docs,
      embeddings = embeddings,
      metadatas = metadatas
  )
  print(f'ChromaDB(VectorDB) 저장 완료. \n현재 문서 수 : {collection.count()}\n')

  print("요약 결과")
  for meta, doc in zip(metadatas, docs):
    print("===" * 20)
    print(f"URL : {meta['url']}")
    print(textwrap.fill(doc, width=70))
    print()


In [29]:
# RAG 검색 + LLM 최종 답변
def answer_with_rag(query:str, top_k:int=3):
  print("\n" + "***" * 20)
  print(f"[질문] : {query}")

  q_vec = embedder.encode([query])[0].tolist()

  # Chroma에서 질문과 유사한 문서를 검색
  result = collection.query(
      query_embeddings=q_vec,
      n_results=top_k,
      include=['documents', 'metadatas', 'distances']
  )

  docs = result['documents'][0]
  metas = result['metadatas'][0]
  dists = result['distances'][0]

  # LLM에게 넘길 context 블록
  context_blocks = []

  print('\n[검색결과]')
  for i, (doc, meta, dist) in enumerate(zip(docs, metas, dists), 1):
    url = meta['url'] # 문서 출처 URL
    print('---' * 30)
    print(f'{i}, URL:{url}')
    print(f'  distance:{dist:.4f}')
    print(textwrap.fill(doc, width=70))

    context_blocks.append(f'[{i}] 출처 : {url}\n{doc}')

  context = '\n\n'.join(context_blocks)

  # LLM에게 보낼 최종 RAG Prompt
  prompt = f"""
  너는 RAG 기반 한국어 전문 도우미야.
  아래 검색된 문서 요약을 근거로 사용자 질문에 답을 해줘.
  컨텍스트에 없는 내용은 지어내지 말고 말해.

  [사용자 질문]
  {query}

  [검색된 컨텍스트]
  {context}

  [답변]

  """

  response = client.responses.create(
      model = GPT_MODEL,
      input = prompt,
      temperature=0.3
  )
  print('\n최종답변')
  print(textwrap.fill(response.output_text.strip(), width=70))

In [30]:
# 실행
await build_db_async()  # 웹 문서를 읽어 LLM에게 요약하게 한 뒤 VectorDB에 저장하는 작업을 비동기 처리

# 질문
questions = [
    "김치찌개의 특징과 재료를 설명해줘",
    "인공지능이 뭐야?",
    "LOL의 전체적인 전망은 어떻게 돼?"
  ]

for q in questions:
  answer_with_rag(q)


[FETCH] https://ko.wikipedia.org/wiki/김치찌개 -> 200
[FETCH] https://ko.wikipedia.org/wiki/인공지능 -> 200


/usr/lib/python3.12/html/parser.py:425: RuntimeWarning: coroutine 'hello' was never awaited
  m = attrfind_tolerant.match(rawdata, k)


[FETCH] https://ko.wikipedia.org/wiki/LCK -> 200
ChromaDB(VectorDB) 저장 완료. 
현재 문서 수 : 3

요약 결과
URL : https://ko.wikipedia.org/wiki/김치찌개
김치찌개는 한국의 대표적인 국물 요리로, 김치를 주재료로 하여 얼큰하게 끓인 찌개이다. 이 요리는 고추, 배추, 소금,
고춧가루 등을 사용하여 만들어지며, 김치가 시어지거나 양을 늘리기 위해 물을 넣어 끓이는 방식에서 유래되었다. 궁중 요리에서도
'김치조치'라는 이름으로 소개되었다. 조리법이 간단하여 가정에서 자주 만들어 먹으며, 주로 배추김치, 두부, 육류(주로
돼지고기)와 해산물(참치, 고등어 등)을 사용한다. 신 김치를 사용하면 깊은 맛이 나며, 볶은 후 끓이면 더욱 풍미가
증가한다. 잡내를 줄이기 위해 된장이나 청주를 추가하기도 하며, 마늘과 양파가 도움이 된다. 영양 면에서는 다양한 재료를
추가해 영양소를 보충할 수 있지만, 나트륨과 지방 함량이 높아 칼로리가 높은 편이다. 김치찌개는 밥과 함께 곁들여 먹는 경우가
많고, 다양한 재료를 활용해 여러 형태로 조리할 수 있다.

URL : https://ko.wikipedia.org/wiki/인공지능
인공지능(AI)은 인간의 학습, 추론, 지각 능력을 컴퓨터로 구현하려는 컴퓨터 과학의 한 분야이다. 이는 자연 지능과
구별되며, 인간의 지능을 모방한 시스템으로 정의된다. 초기 정의는 다트머스 회의에서 제안되었으며, 인공지능은 일반적으로 약한
인공지능(weak AI)과 강한 인공지능(strong AI)으로 나뉜다. 약한 인공지능은 특정 문제 해결에 중점을 두고, 강한
인공지능은 인간처럼 사고하고 문제를 해결할 수 있는 능력을 목표로 한다. AGI(인공 일반 지능)의 구현은 추론, 문제 해결,
학습 등의 지능적 특성을 통합해야 한다. 인공지능 연구는 심리학적 접근과 언어 지능의 이해를 목표로 하며, 로보틱스와 집합적
지식도 포함된다. 초기 연구는 다양한 과학적 배경에서 시